# 05 — Model Monitoring with Evidently AI

Detects data drift and model performance degradation between training and test distributions.
Simulates a production monitoring workflow where a deployed model is evaluated against new incoming data.

In [3]:
import os
import pandas as pd
import numpy as np
import evidently
from evidently import Dataset, DataDefinition, BinaryClassification, Report
from sklearn.model_selection import train_test_split

print(f"Evidently version: {evidently.__version__}")

# Load data
features = pd.read_parquet('../data/features.parquet')
test_preds = pd.read_parquet('../data/test_predictions.parquet')

# Recreate train/test split
X = features.drop(columns=['default'])
y = features['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build reference and current dataframes
reference_df = X_train.copy()
reference_df['target'] = y_train.values

current_df = X_test.copy()
current_df['target'] = y_test.values
current_df['pred_proba'] = test_preds['pred_proba'].values
current_df['prediction'] = (current_df['pred_proba'] >= 0.49).astype(int)

print(f"Reference (train): {reference_df.shape}")
print(f"Current (test):    {current_df.shape}")
print(f"\nAvailable top-level Evidently: {[x for x in dir(evidently) if not x.startswith('_')]}")

Evidently version: 0.7.21
Reference (train): (299184, 19)
Current (test):    (74797, 21)

Available top-level Evidently: ['BinaryClassification', 'ColumnType', 'DataDefinition', 'Dataset', 'LLMClassification', 'MulticlassClassification', 'Recsys', 'Regression', 'Report', 'Run', 'compare', 'core', 'descriptors', 'errors', 'generators', 'guardrails', 'legacy', 'llm', 'metrics', 'presets', 'pydantic_utils', 'sdk', 'tests', 'ui', 'utils', 'version_info']


## 1. Define Dataset Schema and Run Drift Report

In [6]:
from evidently.presets import DataDriftPreset
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataSummaryPreset

# Define column types
cat_cols = ['businesstype', 'naics_sector', 'borrstate', 'fixedorvariableinterestind']
num_cols = [c for c in X_train.columns if c not in cat_cols]

# Define data schema
data_def = DataDefinition(
    numerical_columns=num_cols,
    categorical_columns=cat_cols,
)

# Create Evidently datasets
ref_dataset = Dataset.from_pandas(reference_df, data_definition=data_def)
cur_dataset = Dataset.from_pandas(current_df.drop(columns=['pred_proba','prediction']),
                                  data_definition=data_def)

# Run data drift report
drift_report = Report(metrics=[DataDriftPreset()])
my_run = drift_report.run(reference_data=ref_dataset, current_data=cur_dataset)

# Save HTML report
os.makedirs('../reports', exist_ok=True)
my_run.save_html('../reports/data_drift_report.html')
print("Drift report saved to ../reports/data_drift_report.html")

Drift report saved to ../reports/data_drift_report.html


## 2. Extract Drift Metrics

In [9]:
# Extract drift results from metric_results
metric_results = drift_dict.get('metric_results', {})

print("=== Data Drift Report Summary ===\n")

for metric_id, result in metric_results.items():
    display_name = result.get('display_name', 'Unknown')
    # Extract scalar values
    for key in ['count', 'share', 'value']:
        if key in result and isinstance(result[key], dict):
            val = result[key].get('value', 'N/A')
            print(f"  {display_name} ({key}): {val}")
        elif key in result:
            print(f"  {display_name}: {result[key]}")

print(f"\nInterpretation:")
print(f"  0 drifted columns out of 18 features — expected for a random train/test split.")
print(f"  In production, this report would run monthly on live scored loans.")
print(f"  Drift threshold: >50% of features drifting triggers model retraining alert.")

=== Data Drift Report Summary ===

  Count of Drifted Columns (count): 0.0
  Count of Drifted Columns (share): 0.0
  Value drift for log_grossapproval: 0.006772128364933658
  Value drift for guarantee_ratio: 0.004083899903921018
  Value drift for term_years: 0.0071557470835816525
  Value drift for initialinterestrate: 0.008235633331291063
  Value drift for collateralind: 0.0023443423987124334
  Value drift for jobssupported: 0.004780950266845756
  Value drift for is_new_business: 0.0008500290732082605
  Value drift for approvalfy: 0.00356684916429968
  Value drift for fed_funds_rate: 0.003996569310514881
  Value drift for gdp_growth_pct: 0.005398226492128549
  Value drift for baa_credit_spread: 0.006219207640317948
  Value drift for unemployment_rate: 0.006906387170973311
  Value drift for log_bank_assets: 0.003195086907138989
  Value drift for bank_matched: 0.000942355889134844
  Value drift for businesstype: 0.0012017041389954229
  Value drift for naics_sector: 0.008190258484401828
 

## 3. Monitoring Summary

In [ ]:
print("=== Monitoring Summary ===\n")
print(f"  Drifted columns:     0 / 18  (0.00%)")
print(f"  Max drift score:     {max([0.008931, 0.008235, 0.008190]):.4f} (borrstate)")
print(f"  Drift threshold:     0.05 (Wasserstein/PSI depending on column type)")
print(f"  Status:              NO DRIFT DETECTED\n")
print("HTML report saved: ../reports/data_drift_report.html")
print("\nProduction workflow this simulates:")
print("  1. Score incoming loans monthly with deployed model")
print("  2. Run Evidently drift report: new loans vs training data")
print("  3. If >50% features drift OR AUROC drops >0.05 → trigger retraining")
print("  4. Log results to MLflow for trend tracking")
print("\nNext: 06_fairness.ipynb — bias audit and demographic disparity analysis")